In [1]:
from scipy import sparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import os
os.environ["MKL_THREADING_LAYER"] = "GNU"

In [2]:
X = sparse.load_npz("../datasets/clusterization/train.npz")

In [3]:
print(f"Размер: {X.shape}")
M, N = X.shape
print(X[:1, :].toarray().max(), X[:1, :].toarray().min(), X[:1, :].toarray().mean())

Размер: (21000, 3049)
7598.588105474598 -13.815510557964274 2.568935238165224


In [4]:
all_lens = []
for i in range(M):
    row = X[i].data
    all_lens.append(len(row.tolist()))
arr = np.array(all_lens)
values, counts = np.unique(arr, return_counts=True) 
print(f"Мода по размерности: {values[np.argmax(counts)]}")

Мода по размерности: 554


### <div align="center">PCA</div>

In [5]:
def plot_the_PCA_interval(pca_obj, start, stop, step, show_ratios=False, print_cum=False):
    interval = np.arange(start, stop, step)

    explained_variance_ratio = pca_obj.explained_variance_ratio_[interval]
    explained_variance_ratio_reduced = explained_variance_ratio.copy()[1::]
    rations = np.insert(explained_variance_ratio[:-1:]/explained_variance_ratio_reduced, 0, 0)
    cumulative = np.cumsum(pca_obj.explained_variance_ratio_)[interval]
    components = np.arange(0, len(explained_variance_ratio))*step + start

    plt.figure(figsize=(8, 5))
    plt.bar(components, explained_variance_ratio, alpha=0.7, label='Individual')
    plt.plot(components, cumulative, 'ro-', label='Cumulative')
    plt.xlabel('Number of Components')
    plt.ylabel('Explained Variance Ratio')
    plt.xticks(components)
    plt.legend()
    plt.grid(True)
    plt.show()

    if show_ratios or print_cum:
        for comp, cum, ratio in zip(components, cumulative, rations):
            string = f"Компонент {comp}: "
            if show_ratios: string += f"cum={cum:.4f}; "
            if print_cum: string += f"ratio={ratio:.4f}; "
            print(string)

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [7]:
scaler = StandardScaler(with_mean=False)
X = scaler.fit_transform(X)

In [8]:
# pca = PCA()
# pca.fit(X)

In [9]:
# # на 2450 объяснена вся дисперсия (с точностью до 1e-8)
# start, stop, step = 2300, 2500, 10
# plot_the_PCA_interval(pca, start, stop, step, show_ratios=True, print_cum=True)

In [10]:
# start, stop, step = 0, 2450, 100
# plot_the_PCA_interval(pca, start, stop, step, show_ratios=True, print_cum=True)

### Conclusion (PCA):
Можно просто урезать размерность до 2450 компонент не теряя дисперсии, попробуем его применить перед t-sne и umap

In [11]:
pca_new = PCA(n_components=2450)
X_embedded_pca = pca_new.fit_transform(X)

### <div align="center">t-SNE</div>

In [12]:
def plot_3D(X, colors=None, title=None, size=2):    
    fig = px.scatter_3d(
        x=X[:, 0], 
        y=X[:, 1], 
        z=X[:, 2] if X.shape[1] >= 3 else np.zeros(X.shape[0]),
        color=colors,
        title=title,
        size=[size for i in range(X.shape[0])],
        hover_data=None,  # убираем дополнительные данные
        hover_name=None
    )  # убираем имя при наведении
    fig.write_html(f"temp_3d_plot_{title}.html")
    fig.show()


In [13]:
from sklearn.manifold import TSNE

In [14]:
# # попробуем применить к сырым данным
# X_embedded_tsne = TSNE(n_components=3, 
#                   init='random').fit_transform(X[:1000])

In [15]:
# plot_3D(X_embedded_tsne, title="t-SNE")

In [16]:
# X_embedded_pca_tsne = TSNE(n_components=3, 
#                   init='random').fit_transform(X_embedded_pca)

In [17]:
# plot_3D(X_embedded_pca_tsne, title="PCA + t-SNE")

### <div align=center> UMAP </div>

In [18]:
import umap as up

In [19]:
# umap = up.UMAP(n_components=2, n_neighbors=2)
# X_embedded_pca_umap = umap.fit_transform(X_embedded_pca)
# plot_3D(X_embedded_pca_umap, title="PCA+UMAP_n2")

In [20]:
umap = up.UMAP(n_components=3, n_neighbors=2)
X_embedded_pca_umap = umap.fit_transform(X_embedded_pca)
plot_3D(X_embedded_pca_umap, title="PCA+UMAP_n2_2")

c:\Users\user_1\miniforge3\envs\ipynb_VK\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


### <div align="center">Clusterization</div>

In [21]:
from sklearn.cluster import HDBSCAN
import time
import seaborn as sns

In [22]:
def plot_clusters(data, algorithm, args, kwds):
    start_time = time.time()
    labels = algorithm(*args, **kwds).fit_predict(data)
    subm = pd.DataFrame({"ID": np.arange(labels.size), "TARGET": labels})
    subm.to_csv(f"subm_{algorithm.__name__}.csv", index=False)
    print(labels[:10])
    end_time = time.time()
    palette = sns.color_palette('deep', np.unique(labels).max() + 1)
    colors = [palette[x] if x >= 0 else (0.0, 0.0, 0.0) for x in labels]
    plot_3D(data, colors, title=f"{algorithm.__name__}")
    print('Clusters found by {}'.format(str(algorithm.__name__)))
    print('Clustering took {:.2f} s'.format(end_time - start_time))

In [23]:
# plot_clusters(X_embedded_pca_umap,
#               HDBSCAN, 
#               [], 
#               {'min_cluster_size': 150}
# )

HDBSCAN - очень плохие метрики на тестовых данных

#### k-MEANS

In [24]:
# import sklearn.cluster as cluster

# plot_clusters(X_embedded_pca_umap, cluster.KMeans, (), {'n_clusters':100})

k-MEANS - ещё хуже. Но тут можно попробовать сделать перебор кол-ва кластеров, но для этого сначала надо реализовать Silhouette

#### AgglomerativeClustering

In [25]:
# from sklearn.cluster import AgglomerativeClustering

# plot_clusters(X_embedded_pca_umap, AgglomerativeClustering, (), {'n_clusters': 60, 'linkage': 'ward'})

#### SpectralClustering

In [26]:
# from sklearn.cluster import SpectralClustering
# from scipy import sparse
# import numpy as np
# import pandas as pd


# X = sparse.load_npz("../datasets/clusterization/train.npz")
# labels = SpectralClustering(n_clusters = 60, n_neighbors = 10).fit_predict(X)
# subm = pd.DataFrame({"ID": np.arange(labels.size), "TARGET": labels})
# subm.to_csv(f"subm_{SpectralClustering.__name__}.csv", index=False)


### GaussianMixture

In [33]:
from sklearn.mixture import GaussianMixture

plot_clusters(X_embedded_pca_umap[:1000], GaussianMixture, (), {'n_components': 8})

c:\Users\user_1\miniforge3\envs\ipynb_VK\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


[5 3 2 3 7 0 3 1 0 6]


Clusters found by GaussianMixture
Clustering took 0.16 s
